In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
import openpyxl
from sklearn.preprocessing import StandardScaler
import re
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from statsmodels.stats.outliers_influence import variance_inflation_factor


path1 = r"data/raw_data/clinical_train.csv"
path2 = r"data/raw_data/molecular_train.csv"
path3 = r"data/raw_data/target_train.csv"

df_clinical = pd.read_csv(path1, header=0, sep=',')
df_molecular = pd.read_csv(path2, header=0, sep=',')
df_target = pd.read_csv(path3, header=0, sep=',')

FileNotFoundError: [Errno 2] No such file or directory: 'clinical_train.csv'

# Modification clinical

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn import preprocessing   # pour MinMaxScaler

# =====================================================================
# 0. INITIALISATION
# =====================================================================
df_clinical = df_clinical.copy()        # pour ne pas modifier l’original

# =====================================================================
# 1. BM_BLAST
# =====================================================================
df_clinical['BM_BLAST_missing'] = df_clinical['BM_BLAST'].isna().astype(int)
med_bm = df_clinical['BM_BLAST'].median()
df_clinical['BM_BLAST_imp']  = df_clinical['BM_BLAST'].fillna(med_bm)

# BLAST_FLAG
df_clinical['BLAST_FLAG'] = (df_clinical['BM_BLAST_imp'] > 20).astype(int)

df_clinical['BM_BLAST_log']  = np.log1p(df_clinical['BM_BLAST_imp'])
scaler_bm = preprocessing.MinMaxScaler()
df_clinical['BM_BLAST_feat'] = scaler_bm.fit_transform(df_clinical[['BM_BLAST_log']])
# ---------------------------------------------------------
# pas de drop ici : on garde BM_BLAST_imp pour BLOOD_SEVERITY
# ---------------------------------------------------------

# =====================================================================
# 2. WBC
# =====================================================================
df_clinical['WBC_missing'] = df_clinical['WBC'].isna().astype(int)
med_wbc = df_clinical['WBC'].median()
df_clinical['WBC_imp']     = df_clinical['WBC'].fillna(med_wbc)

df_clinical['WBC_log']  = np.log1p(df_clinical['WBC_imp'])
scaler_wbc = preprocessing.MinMaxScaler()
df_clinical['WBC_feat'] = scaler_wbc.fit_transform(df_clinical[['WBC_log']])

# =====================================================================
# 3. ANC
# =====================================================================
df_clinical['ANC_missing'] = df_clinical['ANC'].isna().astype(int)
med_anc = df_clinical['ANC'].median()
df_clinical['ANC_imp']     = df_clinical['ANC'].fillna(med_anc)

df_clinical['ANC_RISK_GROUP'] = pd.cut(
    df_clinical['ANC_imp'],
    bins=[-np.inf, 0.5, 1.5, np.inf],
    labels=['severe', 'moderate', 'normal']
)

df_clinical['ANC_log']  = np.log1p(df_clinical['ANC_imp'])
scaler_anc = preprocessing.MinMaxScaler()
df_clinical['ANC_feat'] = scaler_anc.fit_transform(df_clinical[['ANC_log']])

# =====================================================================
# 4. MONOCYTES
# =====================================================================
df_clinical['MONOCYTES_missing'] = df_clinical['MONOCYTES'].isna().astype(int)
med_mono = df_clinical['MONOCYTES'].median()
df_clinical['MONOCYTES_imp'] = df_clinical['MONOCYTES'].fillna(med_mono)

# MONO_ABSOLU
df_clinical['MONO_ABSOLU'] = df_clinical['MONOCYTES_imp'] * med_wbc

df_clinical['MONOCYTES_log']  = np.log1p(df_clinical['MONOCYTES_imp'])
scaler_mono = preprocessing.MinMaxScaler()
df_clinical['MONOCYTES_feat'] = scaler_mono.fit_transform(df_clinical[['MONOCYTES_log']])


# =====================================================================
# 5. HB
# =====================================================================
df_clinical['HB_missing'] = df_clinical['HB'].isna().astype(int)
med_hb = df_clinical['HB'].median()
df_clinical['HB_imp'] = df_clinical['HB'].fillna(med_hb)

scaler_hb = preprocessing.MinMaxScaler()
df_clinical['HB_feat'] = scaler_hb.fit_transform(df_clinical[['HB_imp']])
# ---------------------------------------------------------
# pas de drop pour HB_imp avant BLOOD_SEVERITY
# ---------------------------------------------------------

# =====================================================================
# 6. PLT
# =====================================================================
df_clinical['PLT_missing'] = df_clinical['PLT'].isna().astype(int)
med_plt = df_clinical['PLT'].median()
df_clinical['PLT_imp'] = df_clinical['PLT'].fillna(med_plt)

df_clinical['PLT_CATEGORY'] = pd.cut(
    df_clinical['PLT_imp'],
    bins=[-np.inf, 50, 100, 150, np.inf],
    labels=['severe', 'moderate', 'low-normal', 'normal']
)

df_clinical['PLT_log']  = np.log1p(df_clinical['PLT_imp'])
scaler_plt = preprocessing.MinMaxScaler()
df_clinical['PLT_feat'] = scaler_plt.fit_transform(df_clinical[['PLT_log']])
# ---------------------------------------------------------
# pas de drop pour PLT_imp avant BLOOD_SEVERITY
# ---------------------------------------------------------

# =====================================================================
# 7. BLOOD_SEVERITY_FLAG  (avant de supprimer *_imp)
# =====================================================================
df_clinical['BLOOD_SEVERITY_FLAG'] = (
    (df_clinical['HB_imp']  < 8) |
    (df_clinical['PLT_imp'] < 50) |
    (df_clinical['ANC_imp'] < 0.5)
).astype(int)



# =====================================================================
# 8. CYTOGENETICS
# =====================================================================
def extract_abnormalities(cyto):
    if pd.isna(cyto):
        return []
    patterns = [r'del\(\d+\)', r't\(\d+;\d+\)', r'\+\d+', r'-\d+', r'monosomy\s*\d+']
    abn = []
    for p in patterns:
        abn.extend(re.findall(p, cyto))
    return abn

def high_risk_marker(anoms):
    markers = ['del(7)', '-7', 'monosomy7', 'del(5)', '-5']
    s = ''.join(anoms).lower().replace(' ', '')
    return int(any(m.lower() in s for m in markers))

df_clinical['CYTO_missing'] = df_clinical['CYTOGENETICS'].isna().astype(int)
df_clinical['cyto_list']    = df_clinical['CYTOGENETICS'].fillna('').apply(extract_abnormalities)
df_clinical['cyto_n_abn']   = df_clinical['cyto_list'].apply(len)
df_clinical['cyto_complex'] = (df_clinical['cyto_n_abn'] >= 3).astype(int)
df_clinical['cyto_highrisk'] = df_clinical['cyto_list'].apply(high_risk_marker)

# CYTO_LOSS_GAIN_BALANCE
df_clinical['cyto_loss_gain_balance'] = (
    df_clinical['cyto_list']
      .apply(lambda lst: sum(1 for x in lst if re.match(r'-\d+', x)) -
                         sum(1 for x in lst if re.match(r'\+\d+', x)))
)


# =====================================================================
# 9. CENTER
# =====================================================================
N = 10
freq = df_clinical['CENTER'].value_counts()
top_centers = freq.nlargest(N).index
df_clinical['CENTER_mod'] = df_clinical['CENTER'].where(
    df_clinical['CENTER'].isin(top_centers), 'Other'
)

center_ohe = pd.get_dummies(df_clinical['CENTER_mod'], prefix='center', drop_first=True)
df_clinical['center_freq'] = df_clinical['CENTER'].map((freq / len(df_clinical)).to_dict())

# CENTER_size_decile (1–10)
df_clinical['CENTER_size_decile'] = pd.qcut(df_clinical['center_freq'], 10, labels=False, duplicates='drop') + 1

df_clinical = pd.concat([df_clinical, center_ohe], axis=1)

# =====================================================================
# 10. FEATURES COMBINÉES
# =====================================================================

# PANCYTOPENIA
df_clinical['PANCYTOPENIA'] = (
    (med_hb  < 10) &
    (med_plt < 100) &
    (med_wbc < 4)
).astype(int)

# ANC / WBC
df_clinical['ANC_WBC_RATIO'] = df_clinical['ANC_feat'] / (df_clinical['WBC_feat'] + 1e-5)

# MONO / WBC
df_clinical['MONO_WBC_RATIO'] = df_clinical['MONOCYTES_feat'] / (df_clinical['WBC_feat'] + 1e-5)

# BLAST_PLT_INTERACT
df_clinical['BLAST_PLT_INTERACT'] = df_clinical['BM_BLAST_feat'] * df_clinical['PLT_feat']

# IPSS_like_score
df_clinical['IPSS_like_score'] = (
      2 * ((df_clinical['BLAST_FLAG'] == 1) | (df_clinical['cyto_complex'] == 1) | (df_clinical['cyto_highrisk'] == 1))
    + 1 * ((df_clinical['BLAST_FLAG'] == 0) & (df_clinical['cyto_n_abn'] > 0) & (df_clinical['cyto_highrisk'] == 0))
).astype(int)
df_clinical.drop(['CENTER','CENTER_mod'], axis=1, inplace=True)
df_clinical.drop(['CYTOGENETICS','cyto_list'], axis=1, inplace=True)
# Drop des *_imp désormais inutiles
df_clinical.drop(['HB', 'HB_imp', 'PLT', 'PLT_imp',
                  'BM_BLAST_imp', 'WBC_imp', 'ANC_imp', 'MONOCYTES_imp'], axis=1, inplace=True)
df_clinical.drop(['ANC', 'ANC_log'], axis=1, inplace=True)   # conserve ANC_imp
df_clinical.drop(['MONOCYTES', 'MONOCYTES_log'], axis=1, inplace=True)  # conserve MONOCYTES_imp
df_clinical.drop(['WBC', 'WBC_log'], axis=1, inplace=True)   # conserve WBC_imp
# =====================================================================
# FIN – df_clinical contient désormais toutes les nouvelles features
# =====================================================================
print("Shape final :", df_clinical.shape)
print(df_clinical.head())


# Modification molecular

In [ ]:
##############################################################################
# 0. IMPORTS (si pas déjà faits plus haut)
##############################################################################
import re
import numpy as np
import pandas as pd
from sklearn import preprocessing

##############################################################################
# 1. -----------------------------  MOLECULAR  ------------------------------
#    Toute la préparation ligne-par-ligne ↓
##############################################################################

# ------------------------------------------------------------------  
# 1-A) Longueur de mutation (déjà présent)                      
# ------------------------------------------------------------------  
df_molecular['mut_length']           = df_molecular['END'] - df_molecular['START']
df_molecular['mut_length_missing']   = df_molecular['mut_length'].isna().astype(int)
df_molecular['mut_length_imp']       = df_molecular['mut_length'].fillna(0).clip(lower=0)
df_molecular['mut_length_log']       = np.log1p(df_molecular['mut_length_imp'])
scaler_len = preprocessing.MinMaxScaler()
df_molecular['mut_length_feat']      = scaler_len.fit_transform(df_molecular[['mut_length_log']])
df_molecular.drop(['START','END','mut_length','mut_length_imp','mut_length_log'], axis=1, inplace=True)

# ------------------------------------------------------------------  
# 1-B) REF / ALT : longueurs, GC, indels                         
# ------------------------------------------------------------------  
df_molecular['ref_len']   = df_molecular['REF'].fillna('').apply(len)
df_molecular['alt_len']   = df_molecular['ALT'].fillna('').apply(len)
df_molecular['is_indel']  = (df_molecular['ref_len'] != df_molecular['alt_len']).astype(int)
df_molecular['len_ratio'] = df_molecular['alt_len'] / (df_molecular['ref_len'] + 1)

def gc_content(seq: str) -> float:
    if not seq:
        return 0.0
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

df_molecular['ref_gc'] = df_molecular['REF'].fillna('').apply(gc_content)
df_molecular['alt_gc'] = df_molecular['ALT'].fillna('').apply(gc_content)

cont_feats = ['ref_len','alt_len','len_ratio','ref_gc','alt_gc']
scaler_cont =  preprocessing.MinMaxScaler()
df_molecular[[f'{f}_std' for f in cont_feats]] = scaler_cont.fit_transform(df_molecular[cont_feats])
df_molecular.drop(['REF','ALT'] + cont_feats, axis=1, inplace=True)

# ------------------------------------------------------------------  
# 1-C) CHR one-hot                                                
# ------------------------------------------------------------------  
df_molecular['CHR'] = df_molecular['CHR'].fillna('Unknown')
df_molecular = pd.get_dummies(df_molecular, columns=['CHR'], prefix='chr', drop_first=True)

# ------------------------------------------------------------------  
# 1-D) --------   GENE dummies + FEATURES patient-level   --------  
# ------------------------------------------------------------------  
top_k      = 50
top_genes  = df_molecular['GENE'].value_counts().nlargest(top_k).index.tolist()
df_molecular['GENE_mod'] = df_molecular['GENE'].where(df_molecular['GENE'].isin(top_genes), other='Other')

gene_dummies = pd.get_dummies(df_molecular['GENE_mod'], prefix='gene').astype(int)
gene_features = (pd.concat([df_molecular[['ID']], gene_dummies], axis=1)
                   .groupby('ID').max()
                   .reset_index())        # 0/1 présence par gène

# ------------------------------------------------------------------  
# 1-E)  INDICATEURS NUMÉRIQUES patient-level AVANT drop            
# ------------------------------------------------------------------  

# a) n_mut_total
n_mut_total = df_molecular.groupby('ID').size().rename('n_mut_total').reset_index()

# b) n_genes_mut
n_genes_mut = df_molecular.groupby('ID')['GENE'].nunique().rename('n_genes_mut').reset_index()

# c) TP53_high_clonal (VAF > 0.4)  ➜ doit se faire AVANT drop de VAF
df_molecular['TP53_highvaf_tmp'] = (
    (df_molecular['GENE'] == 'TP53') &
    (df_molecular['VAF'].fillna(0) > 0.4)
).astype(int)
tp53_flag = df_molecular.groupby('ID')['TP53_highvaf_tmp'].max().rename('TP53_high_clonal').reset_index()
df_molecular.drop('TP53_highvaf_tmp', axis=1, inplace=True)

# d) clonal_mut (VAF_imp > 0.3)
df_molecular['VAF_imp_tmp'] = df_molecular['VAF'].fillna(df_molecular['VAF'].median())
df_molecular['clonal_mut_tmp'] = (df_molecular['VAF_imp_tmp'] > 0.3).astype(int)
n_clonal_mut = df_molecular.groupby('ID')['clonal_mut_tmp'].sum().rename('n_clonal_mut').reset_index()
df_molecular.drop(['VAF_imp_tmp','clonal_mut_tmp'], axis=1, inplace=True)

# e) sev_mut  (high_impact, itd, ptd) — sera défini quand effect_cat sera créé
# (on calcule plus bas après mapping EFFECT)

# ------------------------------------------------------------------  
# 1-F) PROTEIN_CHANGE → protchg_feats                              
# ------------------------------------------------------------------  
df_molecular['protchg_missing'] = df_molecular['PROTEIN_CHANGE'].isna().astype(int)
df_molecular['PROTEIN_CHANGE_imp'] = df_molecular['PROTEIN_CHANGE'].fillna('Unknown')

def map_prot_change(x: str) -> str:
    x = x.upper()
    if x in {'UNKNOWN', 'P.?'}:             return 'Unknown'
    if 'PTD' in x:                          return 'PTD'
    if '*'  in x:                           return 'Nonsense'
    if re.search(r'FS', x):                 return 'Frameshift'
    if re.search(r'DEL|INS', x):            return 'Indel'
    if re.match(r'^P\.[A-Z]\d+[A-Z]$', x):  return 'Missense'
    return 'Other'

df_molecular['protchg_type'] = df_molecular['PROTEIN_CHANGE_imp'].apply(map_prot_change)
prot_dummies = pd.get_dummies(df_molecular['protchg_type'], prefix='protchg')
protchg_feats = (pd.concat([df_molecular[['ID']], prot_dummies], axis=1)
                   .groupby('ID').max()
                   .reset_index())
df_molecular.drop(['PROTEIN_CHANGE','PROTEIN_CHANGE_imp','protchg_type'], axis=1, inplace=True)

# ------------------------------------------------------------------  
# 1-G) EFFECT  → effect_feats  + n_severe_mut                       
# ------------------------------------------------------------------  
df_molecular['effect_missing'] = df_molecular['EFFECT'].isna().astype(int)
df_molecular['EFFECT_imp']     = df_molecular['EFFECT'].fillna('unknown')

def map_effect(effect: str) -> str:
    e = effect.lower()
    if e in {
        'stop_gained', 'stop_lost', 'stop_retained_variant',
        'frameshift_variant', 'inframe_codon_loss',
        'inframe_codon_gain', 'inframe_variant'
    }:                                        return 'high_impact'
    if e in {
        'non_synonymous_codon', 'splice_site_variant',
        'initiator_codon_change', 'complex_change_in_transcript'
    }:                                        return 'moderate_impact'
    if e == 'synonymous_codon':               return 'low_impact'
    if e in {'3_prime_utr_variant','2kb_upstream_variant'}:
                                              return 'modifier'
    if e in {'itd','ptd'}:                    return e
    return 'other'

df_molecular['effect_cat'] = df_molecular['EFFECT_imp'].apply(map_effect)

# n_severe_mut (high_impact + itd + ptd)
df_molecular['sev_flag_tmp'] = df_molecular['effect_cat'].isin({'high_impact','itd','ptd'}).astype(int)
n_severe_mut = df_molecular.groupby('ID')['sev_flag_tmp'].sum().rename('n_severe_mut').reset_index()
df_molecular.drop('sev_flag_tmp', axis=1, inplace=True)

effect_dummies = pd.get_dummies(df_molecular['effect_cat'], prefix='effect')
effect_feats   = (pd.concat([df_molecular[['ID']], effect_dummies], axis=1)
                    .groupby('ID').max()
                    .reset_index())
df_molecular.drop(['EFFECT','EFFECT_imp','effect_cat'], axis=1, inplace=True)

# ------------------------------------------------------------------  
# 1-H) VAF (arcsin-sqrt)  + drop                                  
# ------------------------------------------------------------------  
df_molecular['VAF_missing'] = df_molecular['VAF'].isna().astype(int)
med_vaf = df_molecular['VAF'].median()
df_molecular['VAF_imp']     = df_molecular['VAF'].fillna(med_vaf)
df_molecular['VAF_asin']    = np.arcsin(np.sqrt(df_molecular['VAF_imp']))
scaler_vaf =  preprocessing.MinMaxScaler()
df_molecular['VAF_feat']    = scaler_vaf.fit_transform(df_molecular[['VAF_asin']])
df_molecular.drop(['VAF','VAF_imp','VAF_asin'], axis=1, inplace=True)

# ------------------------------------------------------------------  
# 1-I) DEPTH (log1p) + drop                                       
# ------------------------------------------------------------------  
df_molecular['DEPTH_missing'] = df_molecular['DEPTH'].isna().astype(int)
med_depth = df_molecular['DEPTH'].median()
df_molecular['DEPTH_imp']  = df_molecular['DEPTH'].fillna(med_depth)
df_molecular['DEPTH_log']  = np.log1p(df_molecular['DEPTH_imp'])
scaler_depth =  preprocessing.MinMaxScaler()
df_molecular['DEPTH_feat'] = scaler_depth.fit_transform(df_molecular[['DEPTH_log']])
df_molecular.drop(['DEPTH','DEPTH_imp','DEPTH_log'], axis=1, inplace=True)

##############################################################################
# 2. ---------------------  AGRÉGATIONS PATIENT-LEVEL  -----------------------
##############################################################################

# indel_ratio
indel_ratio = df_molecular.groupby('ID')['is_indel'].mean().rename('indel_ratio').reset_index()

##############################################################################
# 3. ------------------------  MERGES CIBLÉS  --------------------------------
#    (On ne fait plus de .fillna(0) global sur tout df_clinical)
##############################################################################

def safe_left_merge(base_df, add_df, fill_zero_cols=None):
    """
    Merge à gauche, puis remplit 0 uniquement dans les colonnes indiquées.
    """
    out = base_df.merge(add_df, on='ID', how='left')
    if fill_zero_cols is not None and len(fill_zero_cols):
        out[fill_zero_cols] = out[fill_zero_cols].fillna(0)
    return out

# ---- merge 1 : gènes dummies ----
gene_cols = gene_features.columns.drop('ID')
df_clinical = safe_left_merge(df_clinical, gene_features, gene_cols)

# ---- merge 2 : n_mut_total ----
df_clinical = safe_left_merge(df_clinical, n_mut_total, ['n_mut_total'])

# ---- merge 3 : n_genes_mut ----
df_clinical = safe_left_merge(df_clinical, n_genes_mut, ['n_genes_mut'])

# ---- merge 4 : TP53_high_clonal ----
df_clinical = safe_left_merge(df_clinical, tp53_flag, ['TP53_high_clonal'])

# ---- merge 5 : n_clonal_mut ----
df_clinical = safe_left_merge(df_clinical, n_clonal_mut, ['n_clonal_mut'])

# ---- merge 6 : n_severe_mut ----
df_clinical = safe_left_merge(df_clinical, n_severe_mut, ['n_severe_mut'])

# ---- merge 7 : protchg feats ----
prot_cols = protchg_feats.columns.drop('ID')
df_clinical = safe_left_merge(df_clinical, protchg_feats, prot_cols)

# ---- merge 8 : effect feats ----
effect_cols = effect_feats.columns.drop('ID')
df_clinical = safe_left_merge(df_clinical, effect_feats, effect_cols)

# ---- merge 9 : indel_ratio ----
df_clinical = safe_left_merge(df_clinical, indel_ratio, ['indel_ratio'])


In [ ]:
df_merged = df_clinical.copy()

# LA CELLULE CI DESSOUS EST A EXECUTER UNIQUEMENT POUR LE TRAIN

In [ ]:
''' QUE POUR LE TRAIN, NE PAS EXECUTER QUAND ON FAIT LE TEST
LORSQUE VOUS EXECUTEZ CELLE LA POUR LE TRAIN, NE CONTINUEZ PAS, REMONTEZ POUR FAIRE 
DE MEME POUR LE TEST'''
df_ready_train = pd.merge(df_merged, df_target, on='ID', how='left')

# -----------------------------------------------------------------------------
# 0) POINT DE DÉPART
# -----------------------------------------------------------------------------
df = df_ready_train.copy()          # patient-level
df_molecular_train = df_molecular.copy()   # mutation-level

# -----------------------------------------------------------------------------
# 1) INTERACTIONS CLINIQUES SUPPLÉMENTAIRES
# -----------------------------------------------------------------------------
df['ANC_WBC_ratio']      = df['ANC_feat']      / (df['WBC_feat']  + 1e-6)
df['HB_PLT_ratio']       = df['HB_feat']       / (df['PLT_feat']  + 1e-6)
df['BLAST_WBC_interact'] = df['BM_BLAST_feat'] *  df['WBC_feat']
df['inflammation_sum']   = df[['WBC_feat','ANC_feat','MONOCYTES_feat']].sum(axis=1)
df['inflammation_prod']  = df['WBC_feat'] * df['ANC_feat'] * df['MONOCYTES_feat']

# -----------------------------------------------------------------------------
# 2) PCA + CLUSTERING SUR VARIABLES CLINIQUES
# -----------------------------------------------------------------------------
clin_feats = ['BM_BLAST_feat','WBC_feat','ANC_feat',
              'MONOCYTES_feat','HB_feat','PLT_feat']

pca_clin = PCA(n_components=2, random_state=0)
df[['clin_pca1','clin_pca2']] = pca_clin.fit_transform(df[clin_feats])

km_clin = KMeans(n_clusters=2, random_state=0, n_init='auto')
df['clin_cluster'] = km_clin.fit_predict(df[clin_feats])

# -----------------------------------------------------------------------------
# 3) AGRÉGAT MOLÉCULAIRE (mut_length_feat, VAF_feat, DEPTH_feat)
# -----------------------------------------------------------------------------
# Try to use the correct column names for molecular features
possible_mol_feats = [
    ['mut_length_feat', 'VAF_feat', 'DEPTH_feat'],
    ['mut_length', 'VAF', 'DEPTH'],
    ['mut_length', 'VAF_feat', 'DEPTH_feat'],
    ['mut_length_feat', 'VAF', 'DEPTH'],
]
for feats in possible_mol_feats:
    if all(f in df_molecular_train.columns for f in feats):
        mol_feats = feats
        break
else:
    raise KeyError("Could not find suitable columns for molecular features in df_molecular_train.")

mol_mean = (
    df_molecular_train.groupby('ID')[mol_feats]
      .mean()
      .fillna(0)
      .reset_index()
)

# PCA + clustering
pca_mol = PCA(n_components=2, random_state=0)
mol_mean[['mol_pca1','mol_pca2']] = pca_mol.fit_transform(mol_mean[mol_feats])

km_mol = KMeans(n_clusters=3, random_state=0, n_init='auto')
mol_mean['mol_cluster'] = km_mol.fit_predict(mol_mean[mol_feats])

# merge
def safe_left_merge(base, add, cols):
    out = base.merge(add, on='ID', how='left')
    out[cols] = out[cols].fillna(0)
    return out

df = safe_left_merge(df, mol_mean[['ID','mol_pca1','mol_pca2','mol_cluster']],
                     ['mol_pca1','mol_pca2','mol_cluster'])

# -----------------------------------------------------------------------------
# 4) COMPTE DE MUTATIONS (proxy n_effects)
# -----------------------------------------------------------------------------
n_effects = (
    df_molecular_train.groupby('ID').size()
      .rename('n_effects')
      .reset_index()
)
df = safe_left_merge(df, n_effects, ['n_effects'])

# -----------------------------------------------------------------------------
# 5) SÉLECTION DE VARIABLES (corrélation + VIF)  — PATCH ANTI-NaN
# -----------------------------------------------------------------------------
exclude  = ['ID', 'OS_YEARS', 'OS_STATUS']
num_cols = df.select_dtypes(include=[np.number]).columns
features = [c for c in num_cols if c not in exclude]

# Standardisation
X_scaled = pd.DataFrame(
    StandardScaler().fit_transform(df[features]),
    columns=features
)

# --- PATCH : remove NaN / Inf avant corrélation & VIF ---
X_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)
X_scaled.fillna(0, inplace=True)

# 5.1 Corrélations > .8
corr  = X_scaled.corr().abs()
mask  = np.triu(np.ones_like(corr, dtype=bool), k=1)
high  = corr.where(mask).stack().loc[lambda s: s > .8]

to_drop   = set()
mean_corr = corr.mean()

for f1, f2 in high.index:
    if f1 not in to_drop and f2 not in to_drop:
        to_drop.add(f1 if mean_corr[f1] > mean_corr[f2] else f2)

# 5.2 VIF itératif
remaining = [f for f in features if f not in to_drop]
X_vif = X_scaled[remaining].copy()

def compute_vif(mat):
    return pd.Series(
        [variance_inflation_factor(mat.values, i) for i in range(mat.shape[1])],
        index=mat.columns
    )

while True:
    vifs = compute_vif(X_vif)
    if vifs.max() > 10:
        bad = vifs.idxmax()
        remaining.remove(bad)
        X_vif = X_vif.drop(columns=[bad])
    else:
        break

# -----------------------------------------------------------------------------
# 6) DATAFRAME FINAL
# -----------------------------------------------------------------------------
df_ready_train = df[['ID','OS_YEARS','OS_STATUS'] + remaining].copy()


NameError: name 'df_merged' is not defined

# Pour le test, run toutes les cellules d'en haut en changeant les chemins, SAUF LA CELLULE CI-DESSUS, PUIS CELLE LA 

In [ ]:
df_ready_test = df_merged

# =============================================================================
# ALIGNER LE DATASET TEST SUR LE JEU D’ENTRAÎNEMENT
#   • remaining          : liste définitive des variables retenues sur le train
#   • df_ready_train     : DataFrame train final   (ID + OS_YEARS/OS_STATUS + remaining)
#   • df_test_features   : DataFrame test déjà PASSÉ dans le même pipeline
#                          (une ligne = un patient, sans OS_YEARS / OS_STATUS)
# =============================================================================

# -----------------------------------------------------------------------------
# 1) Ajout des colonnes manquantes (test ou train)
# -----------------------------------------------------------------------------
rem_set = set(remaining)

# 1-a) Si, par exception, une variable de remaining n’est pas dans le train
missing_in_train = rem_set - set(df_ready_train.columns)
for col in missing_in_train:
    df_ready_train[col] = 0.0     # valeur neutre
    print(f"[train] colonne ajoutée : {col}")

# 1-b) Variables absentes dans le test (cas fréquent : dummies jamais vus)
missing_in_test = rem_set - set(df_ready_test.columns)
for col in missing_in_test:
    df_ready_test[col] = 0.0
    print(f"[test ] colonne ajoutée : {col}")

# -----------------------------------------------------------------------------
# 2) Ré-ordonnancement & restriction EXACTE aux variables sélectionnées
# -----------------------------------------------------------------------------
meta_train = ['ID', 'OS_YEARS', 'OS_STATUS']
meta_test  = ['ID']                    # le test n’a pas la cible

df_ready_train = df_ready_train[meta_train + remaining].copy()
df_ready_test  = df_ready_test [meta_test  + remaining].copy()

# -----------------------------------------------------------------------------
# 3) Vérifications rapides
# -----------------------------------------------------------------------------
assert list(df_ready_train.columns[len(meta_train):]) == remaining
assert list(df_ready_test.columns[1:])               == remaining

print("Alignement terminé ↦")
print("df_ready_train :", df_ready_train.shape)
print("df_ready_test  :", df_ready_test.shape)


In [ ]:
df_ready_train
df_ready_test